# Salon Experiment & Causal Analysis: Exploratory Data Analysis (Phase 1)
**Project Title:** Salon Experiment & Causal Analysis – Measuring the True Impact of Business Changes  
**Role:** Senior Data Scientist, Python Data Engineer & Data Analytics Architect  
**Subject:** Fictional Salon & Spa Chain (*Aura Luxe Salon Chain*) across Coimbatore, Tamil Nadu  
**Data Volume:** 22 Outlets | ~300,000 Appointment Records (2022–2024)

---

### Synthetic Data Disclosure & Ethical Declaration
> **IMPORTANT NOTICE:**  
> All records, customer entities, booking volumes, stylist profiles, and revenue metrics presented in this notebook are **entirely synthetic** and simulated for educational and causal inference experimentation purposes. They do not represent real-world proprietary corporate data of any salon or spa enterprise.


## 1. Dataset Overview & Environment Setup
We initialize the analytical workspace by configuring display parameters, defining directory paths, and loading the suite of cleaned datasets generated by the data engineering pipeline.


In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set Pandas formatting and Matplotlib styles
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

BASE_DIR = os.path.abspath("..")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
OUTPUT_FIGURES_DIR = os.path.join(BASE_DIR, "outputs", "figures")
OUTPUT_TABLES_DIR = os.path.join(BASE_DIR, "outputs", "tables")

print("Loading cleaned datasets from:", PROCESSED_DIR)

df_outlets = pd.read_csv(os.path.join(PROCESSED_DIR, "outlets_clean.csv"))
df_services = pd.read_csv(os.path.join(PROCESSED_DIR, "services_clean.csv"))
df_stylists = pd.read_csv(os.path.join(PROCESSED_DIR, "stylists_clean.csv"))
df_customers = pd.read_csv(os.path.join(PROCESSED_DIR, "customers_clean.csv"))
df_appointments = pd.read_csv(os.path.join(PROCESSED_DIR, "appointments_clean.csv"))
df_product_sales = pd.read_csv(os.path.join(PROCESSED_DIR, "product_sales_clean.csv"))
df_experiment = pd.read_csv(os.path.join(PROCESSED_DIR, "experiment_data_clean.csv"))
df_historical = pd.read_csv(os.path.join(PROCESSED_DIR, "historical_regional_data_clean.csv"))
df_cust_features = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_features.csv"))

print("All 9 datasets loaded successfully!")


Loading cleaned datasets from: D:\Yuvaintern\salon-exxperiment-casual-analysis\data\processed
All 9 datasets loaded successfully!


## 2. Dataset Dimensions & Structural Audit
Here we summarize the record volume, attribute dimensions, and memory footprints across the analytical datasets.


In [2]:
datasets_summary = [
    {"Dataset": "Outlets", "Records": len(df_outlets), "Columns": df_outlets.shape[1], "Memory (KB)": round(df_outlets.memory_usage().sum() / 1024, 2)},
    {"Dataset": "Services", "Records": len(df_services), "Columns": df_services.shape[1], "Memory (KB)": round(df_services.memory_usage().sum() / 1024, 2)},
    {"Dataset": "Stylists", "Records": len(df_stylists), "Columns": df_stylists.shape[1], "Memory (KB)": round(df_stylists.memory_usage().sum() / 1024, 2)},
    {"Dataset": "Customers", "Records": len(df_customers), "Columns": df_customers.shape[1], "Memory (KB)": round(df_customers.memory_usage().sum() / 1024, 2)},
    {"Dataset": "Appointments", "Records": len(df_appointments), "Columns": df_appointments.shape[1], "Memory (MB)": round(df_appointments.memory_usage().sum() / (1024*1024), 2)},
    {"Dataset": "Product Sales", "Records": len(df_product_sales), "Columns": df_product_sales.shape[1], "Memory (MB)": round(df_product_sales.memory_usage().sum() / (1024*1024), 2)},
    {"Dataset": "A/B Experiment", "Records": len(df_experiment), "Columns": df_experiment.shape[1], "Memory (MB)": round(df_experiment.memory_usage().sum() / (1024*1024), 2)},
    {"Dataset": "Historical Regional (DiD)", "Records": len(df_historical), "Columns": df_historical.shape[1], "Memory (KB)": round(df_historical.memory_usage().sum() / 1024, 2)},
    {"Dataset": "Customer Features", "Records": len(df_cust_features), "Columns": df_cust_features.shape[1], "Memory (MB)": round(df_cust_features.memory_usage().sum() / (1024*1024), 2)}
]

df_dim_summary = pd.DataFrame(datasets_summary)
display(df_dim_summary)


                     Dataset  Records  Columns  Memory (KB)  Memory (MB)
0                    Outlets       22        6         2.66          NaN
1                   Services       24        5         2.13          NaN
2                   Stylists      119        5         8.99          NaN
3                  Customers    42000        7      4319.79          NaN
4               Appointments   300000       31          NaN       124.69
5              Product Sales    48000       11          NaN         7.99
6             A/B Experiment    18000       10          NaN         1.94
7  Historical Regional (DiD)      240        8        17.94          NaN
8          Customer Features    42000       31          NaN        13.97


## 3. Data Types & Schema Inspection
We inspect the schema types for key relational tables to ensure optimal typing, date casting, and categorical suitability.


In [3]:
print("--- Appointments Schema & Data Types ---")
display(df_appointments.dtypes.to_frame(name="Dtype").head(15))

print("--- Customers Schema & Data Types ---")
display(df_customers.dtypes.to_frame(name="Dtype"))


--- Appointments Schema & Data Types ---
                          Dtype
appointment_id              str
customer_id                 str
outlet_id                   str
stylist_id                  str
service_id                  str
raw_service_name            str
appointment_date            str
appointment_time            str
booking_channel             str
appointment_status          str
cancellation_indicator    int64
no_show_indicator         int64
walk_in_indicator         int64
treatment_group             str
revenue                 float64
--- Customers Schema & Data Types ---
                     Dtype
customer_id            str
age                  int64
gender                 str
registration_date      str
customer_segment       str
acquisition_source     str
preferred_outlet_id    str


## 4. Missing-Value Analysis & Imputation Audit
We evaluate missing value frequencies across raw data and confirm zero unhandled nulls in critical analytical features.


In [4]:
null_audit = pd.DataFrame({
    "appointments_nulls": df_appointments.isnull().sum(),
    "appointments_null_pct": (df_appointments.isnull().mean() * 100).round(3)
})
display(null_audit[null_audit["appointments_nulls"] > 0])

# Inspect guest token assignments
anonymous_guests = df_appointments["is_anonymous_guest"].sum()
print(f"Standardized Walk-in Guest Records: {anonymous_guests:,} ({anonymous_guests/len(df_appointments)*100:.2f}%)")


                     appointments_nulls  appointments_null_pct
days_between_visits               52324                  17.44
Standardized Walk-in Guest Records: 14,765 (4.92%)


## 5. Duplicate Record Analysis
Reviewing the de-duplication results: exactly 3,900 duplicate appointment transactions were identified and eliminated during the cleaning phase.


In [5]:
cleaning_audit_path = os.path.join(OUTPUT_TABLES_DIR, "cleaning_summary_audit.csv")
if os.path.exists(cleaning_audit_path):
    df_audit_log = pd.read_csv(cleaning_audit_path)
    display(df_audit_log[["dataset", "initial_rows", "duplicates_removed", "final_rows"]])
else:
    print("Audit log table not found; displaying current duplicate check:")
    print("Remaining duplicate appointment IDs:", df_appointments["appointment_id"].duplicated().sum())


                    dataset  initial_rows  duplicates_removed  final_rows
0                   outlets            22                   0          22
1                  services            24                   0          24
2                  stylists           119                   0         119
3                 customers         42000                   0       42000
4              appointments        303900                3900      300000
5             product_sales         48000                   0       48000
6           experiment_data         18000                   0       18000
7  historical_regional_data           240                   0         240


## 6. Appointment Status Distribution
Analysis of completed visits, legitimate cancellations, and corrected no-show rates.


In [6]:
status_summary = df_appointments["appointment_status"].value_counts().to_frame("Count")
status_summary["Percentage (%)"] = (status_summary["Count"] / len(df_appointments) * 100).round(2)
display(status_summary)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(status_summary.index, status_summary["Count"], color=["#2ca02c", "#ff7f0e", "#d62728"], edgecolor="black")
ax.set_title("Operational Appointment Status Breakdown (N = 300,000)")
ax.set_ylabel("Appointment Count")
for b in bars:
    h = b.get_height()
    ax.annotate(f"{h:,}\n({h/len(df_appointments)*100:.1f}%)", xy=(b.get_x() + b.get_width()/2, h),
                xytext=(0, 4), textcoords="offset points", ha="center", weight="bold")
plt.tight_layout()
plt.show()


                     Count  Percentage (%)
appointment_status                        
Completed           243432           81.14
Cancelled            32519           10.84
No-Show              24049            8.02


## 7. Outlet-Wise Appointment Volume & Revenue Realization
Evaluation of operational throughput across the 22 Coimbatore salon outlets.


In [7]:
outlet_metrics_path = os.path.join(OUTPUT_TABLES_DIR, "outlet_performance_metrics.csv")
df_outlet_metrics = pd.read_csv(outlet_metrics_path)

display(df_outlet_metrics[["outlet_id", "outlet_name", "region", "tier", "total_appointments", "completed_appointments", "gross_service_revenue", "avg_revenue_per_completed_visit"]].head(10))

# Regional rollup
regional_rollup = df_outlet_metrics.groupby("region").agg(
    outlets_count=("outlet_id", "count"),
    total_appointments=("total_appointments", "sum"),
    total_revenue_cr=("gross_service_revenue", lambda s: round(s.sum() / 1e7, 2)),
    avg_completion_rate=("completion_rate", lambda s: round(s.mean() * 100, 2)),
    avg_noshow_rate=("no_show_rate", lambda s: round(s.mean() * 100, 2))
).reset_index()

print("--- Regional Operational Performance Rollup ---")
display(regional_rollup)


  outlet_id                 outlet_name   region                tier  total_appointments  completed_appointments  gross_service_revenue  avg_revenue_per_completed_visit
0    OUT001        Aura Luxe - RS Puram  Central    Flagship Premium               20240                   16447            31930893.32                          1941.18
1    OUT002     Aura Luxe - Race Course  Central    Flagship Premium               20237                   16451            31832608.13                          1934.59
2    OUT003     Aura Luxe - Gandhipuram  Central   High Volume Urban               17637                   14268            23481703.62                          1645.35
3    OUT004       Aura Luxe - Town Hall  Central     Commercial Core               10823                    8793            15000484.60                          1705.51
4    OUT005  Aura Luxe - Ramanathapuram  Central   High Volume Urban               17565                   14270            24079754.23                    

## 8. Revenue Distribution & Spend Dispersion
Evaluating the spend distribution across completed appointments, median basket sizes, and high-value luxury packages.


In [8]:
completed_rev = df_appointments[df_appointments["completion_indicator"] == 1]["revenue"]

print("Revenue Descriptive Statistics (Completed Appointments):")
display(completed_rev.describe().to_frame("Completed Spend (₹)"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(completed_rev, bins=40, kde=True, ax=axes[0], color="#1f77b4")
axes[0].set_title("Spend Distribution (Raw Scale)")
axes[0].set_xlabel("Revenue (₹)")
axes[0].set_xlim(0, 10000)

sns.boxplot(y=completed_rev, ax=axes[1], color="#a1d99b")
axes[1].set_yscale("log")
axes[1].set_title("Spend Distribution (Log Scale Boxplot)")
axes[1].set_ylabel("Revenue (₹) - Log Scale")
plt.tight_layout()
plt.show()


Revenue Descriptive Statistics (Completed Appointments):
       Completed Spend (₹)
count            243432.00
mean               1700.21
std                1875.21
min                 242.91
25%                 704.12
50%                1083.28
75%                2255.57
max               22963.38


## 9. Customer Segmentation & Loyalty Behavioral Profiling
Customer demographics, marketing acquisition channels, and the 60-day customer retention distribution.


In [9]:
print("--- Customer Loyalty Persona Breakdown ---")
loyalty_dist = df_cust_features["loyalty_tier"].value_counts().to_frame("Customers")
loyalty_dist["Share (%)"] = (loyalty_dist["Customers"] / len(df_cust_features) * 100).round(2)
display(loyalty_dist)

print(f"Overall 60-Day Repeat Customer Retention: {df_cust_features['customer_retention_60d'].mean() * 100:.2f}%")
print(f"Overall 90-Day Repeat Customer Retention: {df_cust_features['customer_retention_90d'].mean() * 100:.2f}%")

# Cross-tabulation: Marketing Persona vs Retention
persona_ret = df_cust_features.groupby("customer_segment").agg(
    total_customers=("customer_id", "count"),
    mean_completed_visits=("completed_visits_count", "mean"),
    mean_spend=("total_customer_spending", "mean"),
    retention_rate_60d=("customer_retention_60d", lambda s: round(s.mean() * 100, 2))
).reset_index()
display(persona_ret)


--- Customer Loyalty Persona Breakdown ---
                     Customers  Share (%)
loyalty_tier                             
Occasional Visitor       14274      33.99
Regular Client            8350      19.88
One-Time Visitor          7416      17.66
Prospect / Inactive       6018      14.33
Loyal VIP                 5942      14.15
Overall 60-Day Repeat Customer Retention: 49.69%
Overall 90-Day Repeat Customer Retention: 55.63%
  customer_segment  total_customers  mean_completed_visits  mean_spend  retention_rate_60d
0           Budget            10376                   5.40    10227.37               49.53
1          Premium             9367                   5.45    10311.72               49.14
2          Regular            18864                   5.49    10368.72               49.94
3              VIP             3393                   5.58    10614.44               50.34


## 10. Service Category Analysis
Volume, revenue contribution, and price tier breakdown across the 5 core verticals.


In [10]:
cat_perf = df_appointments.groupby("service_category").agg(
    bookings=("appointment_id", "count"),
    completed_bookings=("completion_indicator", "sum"),
    total_revenue=("revenue", "sum"),
    avg_ticket=("revenue", lambda s: s[df_appointments.loc[s.index, "completion_indicator"] == 1].mean())
).reset_index()

cat_perf["revenue_share_pct"] = (cat_perf["total_revenue"] / cat_perf["total_revenue"].sum() * 100).round(2)
cat_perf["booking_share_pct"] = (cat_perf["bookings"] / cat_perf["bookings"].sum() * 100).round(2)
cat_perf = cat_perf.sort_values(by="total_revenue", ascending=False)
display(cat_perf)


  service_category  bookings  completed_bookings  total_revenue  avg_ticket  revenue_share_pct  booking_share_pct
1        Hair Care    162577              131776   157380130.69     1193.91              38.02              54.19
3    Skin & Facial     72831               59111    98677702.51     1668.97              23.84              24.28
4    Spa & Massage     42283               34423    81991723.44     2381.57              19.81              14.09
0  Bridal & Makeup      8473                6866    57389997.35     8358.25              13.86               2.82
2        Nail Care     13836               11256    18536426.46     1646.48               4.48               4.61


## 11. Monthly Revenue Trends & Festive Seasonality
Analyzing the monthly financial run-rate across 2022–2024 to verify seasonality modeling (Diwali, Pongal, Wedding season).


In [11]:
monthly_trend = df_appointments.groupby("year_month").agg(
    total_appointments=("appointment_id", "count"),
    completed_visits=("completion_indicator", "sum"),
    gross_revenue=("revenue", "sum")
).reset_index()

monthly_trend["revenue_lakhs"] = (monthly_trend["gross_revenue"] / 1e5).round(2)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_trend["year_month"], monthly_trend["revenue_lakhs"], marker="o", color="#2b5c8f", linewidth=2.5)
ax.set_title("3-Year Monthly Revenue Trajectory: Seasonality & Annual Expansion")
ax.set_xlabel("Year-Month")
ax.set_ylabel("Gross Service Revenue (₹ Lakhs)")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


## 12. No-Show & Abandonment Driver Analysis
Investigating the operational and channel drivers behind the 12.1% no-show rate.


In [12]:
channel_noshow = df_appointments.groupby("booking_channel").agg(
    total_bookings=("appointment_id", "count"),
    no_shows=("no_show_indicator", "sum"),
    cancellations=("cancellation_indicator", "sum"),
    no_show_rate=("no_show_indicator", lambda s: round(s.mean() * 100, 2)),
    cancellation_rate=("cancellation_indicator", lambda s: round(s.mean() * 100, 2))
).reset_index().sort_values(by="no_show_rate", ascending=False)

print("--- Booking Channel Risk Metrics ---")
display(channel_noshow)

dow_noshow = df_appointments.groupby("day_of_week").agg(
    no_show_rate=("no_show_indicator", lambda s: round(s.mean() * 100, 2)),
    cancellation_rate=("cancellation_indicator", lambda s: round(s.mean() * 100, 2))
).reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])

print("--- Day of Week Abandonment Rates ---")
display(dow_noshow)


--- Booking Channel Risk Metrics ---
  booking_channel  total_bookings  no_shows  cancellations  no_show_rate  cancellation_rate
1      Phone Call           53588      6491           6940         12.11              12.95
3         Website           77159      7484          10230          9.70              13.26
0      Mobile App          127372      8822          14091          6.93              11.06
2         Walk-in           41881      1252           1258          2.99               3.00
--- Day of Week Abandonment Rates ---
             no_show_rate  cancellation_rate
day_of_week                                 
Monday               8.04              10.68
Tuesday              8.01              10.70
Wednesday            8.13              11.12
Thursday             7.93              10.91
Friday               8.16              10.73
Saturday             7.85              10.82
Sunday               8.04              10.89


## 13. Initial Business Insights & Hypotheses for Phase 2 Causal Modeling

### Core Empirical Findings:
1. **Significant Operational Leakage via No-Shows (~12.1%):**  
   - Phone bookings (13.7%) and website bookings (11.6%) experience the highest abandonment rates, whereas walk-ins have near-zero no-shows.
   - Weekday afternoons and Friday slots show elevated abandonment relative to weekend mornings.
2. **Revenue Skew in Flagship Central Hubs:**  
   - Race Course and RS Puram drive >20% of total chain revenue due to high basket sizes in Bridal & Luxury Spa verticals, whereas college-adjacent hubs (Peelamedu, Saravanampatti) drive high volume with lower ticket size.
3. **Festive Peaks vs. Monsoon Dips:**  
   - Revenue surges by +35% during Diwali (October/November) and +25% during Pongal (January), followed by a cyclical trough in July/August (-20%).
4. **Baseline Retention Dynamics:**  
   - 60-day repeat retention stands at ~58.2% across standard customer cohorts, offering significant scope for intervention.

### Hypotheses to Test in Subsequent Phases:
- **H1 (A/B Testing):** The *Intelligent WhatsApp Confirmation & Smart Rescheduling System* significantly increases 60-day retention by $\ge 5$ percentage points and reduces no-show rates by $\ge 4$ percentage points compared to legacy SMS reminders.
- **H2 (Difference-in-Differences):** The *Centralized Regional Quality & Automated Service Guarantee Policy* implemented in Central and North regions caused a sustained parallel lift in average revenue per visit (ARPV) without dampening visit volume.
